In [0]:
#Creating medallion pipeline by simulating raw data 
#arriving from multiple sources 



raw_data = [
    (1,  "Alice",  "UAE", 5000, "2024-01-15"),
    (2,  "Bob",    "KSA", 7000, "2024-01-15"),
    (3,  "Carol",  "UAE", 4500, "2024-01-16"),
    (4,  "David",  "UAE", 9000, "2024-01-16"),
    (2,  "Bob",    "KSA", 7000, "2024-01-15"),  # duplicate
    (5,  "Eve",    None,  6000, "2024-01-17"),  # missing country
    (6,  "Frank",  "UAE", None, "2024-01-17")   # missing salary
]

columns = ["id", "name", "country", "salary", "hire_date"]

df_raw = spark.createDataFrame(raw_data, columns)
df_raw.show()
print(f"Total raw rows received: {df_raw.count()}")

+---+-----+-------+------+----------+
| id| name|country|salary| hire_date|
+---+-----+-------+------+----------+
|  1|Alice|    UAE|  5000|2024-01-15|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  3|Carol|    UAE|  4500|2024-01-16|
|  4|David|    UAE|  9000|2024-01-16|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  5|  Eve|   NULL|  6000|2024-01-17|
|  6|Frank|    UAE|  NULL|2024-01-17|
+---+-----+-------+------+----------+

Total raw rows received: 7


In [0]:
#lets say raw data to bronze layers 

df_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_employee")

print(f"Bronze layers created with {df_raw.count()} rows")
print("Duplicates, nulls and all - exactly as recieved")

      

Bronze layers created with 7 rows
Duplicates, nulls and all - exactly as recieved


In [0]:
#lets build silver layers now 

#read from bronze and clean it, like your teradata transformation
#and store procedure
#but now tracked, version and recoverable 

from pyspark.sql.functions import col, to_date

df_silver = spark.read.format("delta").table("bronze_employee")
#step 1 - Remove duplicates on based on id column 
df_silver = df_silver.dropDuplicates(["id"])

#step 2 - remove rows where column are null

df_silver = df_silver.filter(
    col("country").isNotNull() & 
    col("salary").isNotNull()
)

#step3 - convert hire date from string to proper date type

df_silver = df_silver.withColumn(
    "hire_date", to_date(col("hire_date"), "yyyy-MM-dd")
)

df_silver.show()
print(f"Silver layer has {df_silver.count()} clean rows")



+---+-----+-------+------+----------+
| id| name|country|salary| hire_date|
+---+-----+-------+------+----------+
|  1|Alice|    UAE|  5000|2024-01-15|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  3|Carol|    UAE|  4500|2024-01-16|
|  4|David|    UAE|  9000|2024-01-16|
+---+-----+-------+------+----------+

Silver layer has 4 clean rows


In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_employees")

print(f"Silver layer saved with {df_silver.count()} clean rows")
print("Duplicates removed, nulls dropped, dates properly typed")

Silver layer saved with 4 clean rows
Duplicates removed, nulls dropped, dates properly typed


In [0]:
#now we create gold layer for business ready aggregations 
from pyspark.sql.functions import count, avg, max, round 

df_gold = spark.read.format("delta").table("silver_employees")
df_gold_summary = df_gold.groupby("country").agg(
    count("id").alias("total_employee"),
    round(avg("salary"),2).alias("avg_salary"),
    max("salary").alias("max_salary")
)

In [0]:
df_gold_summary.orderBy("total_employee", ascending = False).show()

+-------+--------------+----------+----------+
|country|total_employee|avg_salary|max_salary|
+-------+--------------+----------+----------+
|    UAE|             3|   6166.67|      9000|
|    KSA|             1|    7000.0|      7000|
+-------+--------------+----------+----------+



In [0]:
df_gold_summary.write.format("delta").mode("overwrite").saveAsTable("gold_employee_summary")
print("gold layer saved successfully")
print(f"Gold has {df_gold_summary.count()} country summaries ready for reporting")

gold layer saved successfully
Gold has 2 country summaries ready for reporting


In [0]:
#Final pipeline validation confirmation for all 3 layers existence

print("="*50)
print("Medallion pipeline validation report")
print("=" * 50)

bronze = spark.read.format("delta").table("bronze_employee")
silver = spark.read.format("delta").table("silver_employees")
gold = spark.read.format("delta").table("gold_employee_summary")

print(f"\nBronze layer - raw rows: {bronze.count()}")
print(f"Silver layer - clean rows: {silver.count()}")
print(f"Gold layer - summaries: {gold.count()}")

print("\n-- BRONZE (everything raw)---")
bronze.show()

print("---SILVER (clean and typed) --")
silver.show()

print("--Gold(business ready) ---")
gold.show()

print("="*50)
print("Pipeline completed successfully")
print("=" * 50)

Medallion pipeline validation report

Bronze layer - raw rows: 7
Silver layer - clean rows: 4
Gold layer - summaries: 2

-- BRONZE (everything raw)---
+---+-----+-------+------+----------+
| id| name|country|salary| hire_date|
+---+-----+-------+------+----------+
|  1|Alice|    UAE|  5000|2024-01-15|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  3|Carol|    UAE|  4500|2024-01-16|
|  4|David|    UAE|  9000|2024-01-16|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  5|  Eve|   NULL|  6000|2024-01-17|
|  6|Frank|    UAE|  NULL|2024-01-17|
+---+-----+-------+------+----------+

---SILVER (clean and typed) --
+---+-----+-------+------+----------+
| id| name|country|salary| hire_date|
+---+-----+-------+------+----------+
|  3|Carol|    UAE|  4500|2024-01-16|
|  4|David|    UAE|  9000|2024-01-16|
|  2|  Bob|    KSA|  7000|2024-01-15|
|  1|Alice|    UAE|  5000|2024-01-15|
+---+-----+-------+------+----------+

--Gold(business ready) ---
+-------+--------------+----------+----------+
|country|total_emplo